In [ ]:
import random
import json
import re
import asyncio
import pandas as pd
import os
from tqdm.asyncio import tqdm_asyncio
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_absolute_experiment
from vpei.epistemic_consistency.experiment_utils import print_absolute_experiment_results

input_file = "./data/code_snippets_sample.csv"
df = pd.read_csv(input_file, index_col=0)
df.rename(columns={"code_tokens":"code_snippet"}, inplace=True) # to make clearer prompt templates
# keep only rows where verdict column is "Accepted" or "Runtime Error"
df = df[(df["verdict"]=="Accepted") | (df["verdict"]=="Runtime Error")]
# convert values of "Accepted" in column verdict into "No Runtime Error"
df['verdict'] = df['verdict'].replace('Accepted', 'No Runtime Error')
df 
#make sure ony one unique problem_id per row
df = df.drop_duplicates(subset=['problem_id', 'submission_id'])
df
df

In [ ]:
experiment_name = "code"
system_prompt = EXPERIMENTS[experiment_name]["absolute_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["absolute_experiment"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# model_name = "gpt-4o-mini"
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
# professor_name = "John Smith"
name = "J.S."
code_snippet = df.iloc[2]['code_snippet']
user_prompt = user_prompt_template.format(name=name, political_attitude="Republican", code_snippet=code_snippet)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
response = make_llm_request(model_name, messages, **model_kwargs)
print("Response:", response)

In [ ]:

models = ["gpt-5-mini"]

n = 1
stimuli_factors = ["code_snippet"]
additional_variables_from_df_to_save = ['verdict']
custom_model_kwargs = {}
random_seed = 42
path_to_save_model_outputs = "./absolute_experiment/"


In [ ]:
payloads = await carry_out_absolute_experiment(models=models, df=df, n=n, system_prompt=system_prompt, user_prompt_template=user_prompt_template, stimuli_factors=stimuli_factors, 
                                               additional_variables_from_df_to_save=additional_variables_from_df_to_save, custom_model_kwargs=custom_model_kwargs, 
                                               path_to_save_model_outputs=path_to_save_model_outputs, random_seed=random_seed)

print_absolute_experiment_results(payloads, models)